# Fine-tune a Hebrew NLI model on clean HebNLI  ·  Person B

`nli_rerank.py` defaults to `oriel9p/AlephBERT-FT-HebNLI-LCHAIM`, fine-tuned on all of
HebNLI — including the rows the probe was mined from. It has already seen our
(target, negation) pairs labelled `contradiction`, so its probe scores are partly
recall rather than judgement. This notebook builds the replacement.

**Runs in two places:** the Colab web UI, and VS Code with the Colab extension. In both
the kernel is a remote Google VM — your local files are *not* there, which is why the
first cells clone the repo. Sections 1–2 are CPU-only; the GPU is needed from section 3.

The two environments differ in Colab's *frontend* features — stored secrets, Drive
mounting, browser downloads. Every cell that uses one falls back rather than failing.

Run cell by cell, not Run All. Each cell is preceded by what it should print and what
it leaves behind; if the output disagrees, stop there.

## Where everything ends up

The kernel is a Google VM. Three destinations, and only one survives a runtime reset:

| what | size | where | survives a reset? |
|---|---|---|---|
| cloned repo, `data/raw/*.jsonl` | ~400 MB | VM disk | no |
| smoke-run checkpoint | ~480 MB | VM disk | no |
| full-run model + 2 epoch checkpoints | ~480 MB each | Drive, **if** `--out` points there | yes |
| `results/*.json`, `results/nli_train.csv` | a few KB | VM disk, then downloaded | via git |

Nothing reaches your own machine until the final cell.

## 1. Setup

**Prints** &nbsp; Nothing. This cell only defines a helper.

**Writes** &nbsp; Nothing.

In [ ]:
# Secrets three ways: Colab's store, then the environment, then a prompt. The VS Code
# extension cannot reach Colab's secret store, so the fallbacks are what make this
# notebook portable. getpass also keeps the token out of the saved output.
import os, subprocess, getpass

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f'{name}: from Colab secrets')
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        print(f'{name}: from environment')
        return value.strip()
    return getpass.getpass(f'{name}: ').strip()

**Prints** &nbsp; Python and torch versions, the GPU name, and which `google.colab` modules import. `cuda: True` plus a T4 are the two things to confirm before going further.

**Writes** &nbsp; Nothing.

In [ ]:
# What are we running on? Answers 'will this work here' before anything slow.
import platform
print('python      ', platform.python_version())
print('cwd         ', os.getcwd())
try:
    import torch
    print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu         ', torch.cuda.get_device_name(0))
except ImportError:
    print('torch        not installed yet')
for mod in ('google.colab.userdata', 'google.colab.drive', 'google.colab.files'):
    try:
        __import__(mod)
        print(f'{mod:24s} available')
    except Exception as exc:
        print(f'{mod:24s} NOT available ({type(exc).__name__})')

**Prints** &nbsp; `GH_TOKEN:` and where it came from, pip's log, then the last 3 commits — the top one should be the newest `nli:` commit.

**Writes** &nbsp; The repo at `/content/hebrew-negation-embeddings` on the VM. The working directory moves into it, so every path after this is relative to the repo root.

In [ ]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'main'

gh_token = get_secret('GH_TOKEN')
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else '.')
subprocess.run(['git','pull','-q','origin',BRANCH], check=True)

# drop the token from the stored remote so it is not left on the VM's disk
subprocess.run(['git','remote','set-url','origin',
                f'https://github.com/{OWNER}/{REPO}.git'], check=True)
del gh_token, url

!pip install -q -r requirements.txt
!git log --oneline -3

**Prints** &nbsp; `27/27 checks passed`, then `all pipeline checks passed`, `all projection checks passed`, `all NLI data checks passed`. Anything else: stop here, because everything downstream inherits the fault.

**Writes** &nbsp; Nothing.

In [ ]:
# offline checks first - seconds, no network, no GPU
!python -m src.data.negation --selftest
!python -m tests.test_data_pipeline | tail -3
!python -m tests.test_projection | tail -3
!python -m tests.test_nli_data | tail -3

## 2. Data

HebNLI's repo card marks it private, so the download needs a token; `src/data/hebnli.py`
reads `HF_TOKEN` from the environment.

This has already been run once locally. The cells below should reproduce it exactly —
same code, same data, so a different answer means something is wrong:

| split | loaded | promptID filter | text audit | kept |
|---|---|---|---|---|
| train | 300,067 | −2,068 | −9 | 297,990 |
| val | 1,999 | −9 | −1 | 1,989 |
| test | 884 | −1 | 0 | 883 |

**Prints** &nbsp; `HF_TOKEN:` and where it came from.

**Writes** &nbsp; Nothing. The token stays in memory.

In [ ]:
os.environ['HF_TOKEN'] = get_secret('HF_TOKEN')

**Prints** &nbsp; Four lines per split. Train: `rows 300067`, `prompts 100390`, `prompts with e/n/c 91630`. Then `rows 1999` for val, `rows 884` for test.

**Writes** &nbsp; `data/raw/hebnli_{train,val,test}.jsonl` on the VM, about 400 MB total. Gitignored — regenerate it, never commit it.

In [ ]:
!python -m src.data.hebnli --split train --out data/raw/hebnli_train.jsonl
!python -m src.data.hebnli --split val   --out data/raw/hebnli_val.jsonl
!python -m src.data.hebnli --split test  --out data/raw/hebnli_test.jsonl

**Prints** &nbsp; Per split: `held-out promptIDs 689`, `probe sentences 907`, a three-stage funnel, and the text-overlap count with up to 5 example rows. Train must end at `297990`, val `1989`, test `883`.

**Writes** &nbsp; `data/raw/hebnli_{split}_clean.jsonl` — the training data, on the VM, gitignored.<br>`results/nli_data_{split}.json` — the manifest, a few KB, **committed**. It records every filter count and every text-overlap hit, and is what the report cites.

In [ ]:
# Two filters. The promptID list is Itay's 689 held-out prompts; the text audit catches
# probe sentences reachable under a *different* promptID, which an id filter cannot see.
# All three splits: val shares prompts with train, so an unfiltered val would measure
# validation accuracy on rows the probe itself came from.
# One line each, no loop - IPython rewrites `!` magics line by line, so a backslash
# continuation inside a for-body reaches the Python parser and fails.
!python -m src.nli.prepare_data --source data/raw/hebnli_train.jsonl --split train --out data/raw/hebnli_train_clean.jsonl
!python -m src.nli.prepare_data --source data/raw/hebnli_val.jsonl   --split val   --out data/raw/hebnli_val_clean.jsonl
!python -m src.nli.prepare_data --source data/raw/hebnli_test.jsonl  --split test  --out data/raw/hebnli_test_clean.jsonl

**Prints** &nbsp; Three lines, one per split, matching the table above.

**Writes** &nbsp; Nothing. It only reads the manifests back.

In [ ]:
import json
for split in ('train', 'val', 'test'):
    m = json.load(open(f'results/nli_data_{split}.json', encoding='utf-8'))
    f = m['funnel']
    print(f"{split:6s} loaded={f['loaded']:>7}  id_filter=-{f['loaded']-f['prompt_id_clean']:<5}"
          f"  text_audit=-{m['text_overlap']['rows_dropped']:<3}  kept={m['rows_written']}")

## 3. Smoke run · GPU from here

2000 rows, one epoch, a few minutes. It exercises tokenising, the label map, the
training loop, checkpoint saving and the manifest before hours are committed to any of
them. `train_nli.py` has never run on a GPU or against Colab's transformers version, so
this is where a surprise would surface — cheaply.

Deliberately on the VM's own disk with no epoch checkpoints: the weights are throwaway.

**Prints** &nbsp; The GPU name, then `pair encoding pair_without_segment_ids` (alephbert-base has no segment embeddings), a `[warn] smoke run` line, a progress bar, and `val accuracy` / `val macro F1`. Those two are meaningless at 2000 rows and one epoch, and are flagged `smoke_run` in the results so they cannot be mistaken for findings.

**Writes** &nbsp; `checkpoints/alephbert-hebnli-clean/` on the VM, ~480 MB, throwaway.<br>`results/nli_train_alephbert.json` and one row in `results/nli_train.csv`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

!python -m src.nli.train_nli --base alephbert \
    --train data/raw/hebnli_train_clean.jsonl \
    --val data/raw/hebnli_val_clean.jsonl \
    --max-train 2000 --epochs 1

## 4. Full run

Hours, over ~298k rows. Two things decide whether that survives:

**Where the weights go.** `/content` is wiped when the runtime resets, so Drive is the
only durable option — and mounting Drive is a Colab-frontend feature that may not work
under the VS Code extension. The next cell tries, reports honestly, and picks a path.

**`--save-epochs`.** Writes a resumable checkpoint per epoch, keeping the two newest.
Without it nothing exists on disk until training completes. If the session dies, re-run
the training cell unchanged and it resumes from the newest; `--fresh` starts over.

Swap models with `--base alephbertgimmel`. The key names both the checkpoint directory
and the manifest, so two runs cannot overwrite each other.

**Prints** &nbsp; Either Drive's mount confirmation, or four `[warn]` lines. Either way the last line reads `checkpoint dir: ... | durable: True/False`. **Read that line** — it is the difference between a disconnect costing one epoch and costing the whole run.

**Writes** &nbsp; Mounts Drive at `/content/drive` if it can, and sets `CKPT`. Nothing else.

In [ ]:
# Drive if we can get it, VM disk if we cannot - but say which, loudly, because the
# consequence is not visible until something goes wrong.
CKPT = '/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_DRIVE = True
except Exception as exc:
    ON_DRIVE = False
    CKPT = 'checkpoints/alephbert-hebnli-clean'
    print(f'[warn] Drive unavailable ({type(exc).__name__}: {exc})')
    print('[warn] checkpoints go to the VM disk and die with the runtime.')
    print('[warn] epoch checkpoints still protect against a crashed cell, not a reset.')
    print('[warn] move the finished model off the VM before the session ends.')
print('checkpoint dir:', CKPT, '| durable:', ON_DRIVE)

**Prints** &nbsp; The same shape as the smoke run without the `[warn] smoke run` line, over ~298k rows instead of 2000. `resuming from ...` appears at the top if it picked up a checkpoint. The `val accuracy` here is a real number worth reporting.

**Writes** &nbsp; `$CKPT/` — the final model, tokenizer and `config.json` carrying the label names.<br>`$CKPT/_trainer/checkpoint-N/` — two epoch checkpoints, for resuming only.<br>`results/nli_train_alephbert.json`, and a second row in `nli_train.csv` beside the smoke row rather than replacing it.

In [ ]:
# re-run this exact cell after a disconnect - it resumes from the newest checkpoint
!python -m src.nli.train_nli --base alephbert \
    --train data/raw/hebnli_train_clean.jsonl \
    --val data/raw/hebnli_val_clean.jsonl \
    --save-epochs --out {CKPT}

## 5. Verify, then keep the results

`check_nli_labels` runs six obvious Hebrew pairs and prints the names from
`config.id2label` next to what the model predicts. For the released checkpoint that
discovers an undocumented mapping; for ours it confirms the names we wrote survived
training and describe what the model actually does — a config can say anything.

**Prints** &nbsp; The model path, `encoding pair`, the label names — expect `{0: 'entailment', 1: 'neutral', 2: 'contradiction'}` — then six pairs each marked `[ok]` or `[MISMATCH]`, and a tally. A low tally means the indices are wrong, not the model. The released checkpoint scores 6/6.

**Writes** &nbsp; Nothing.

In [ ]:
!python -m src.interventions.check_nli_labels \
    --model {CKPT} --subfolder ""

**Prints** &nbsp; One row per training configuration — the smoke run and the full run side by side, with `smoke_run` telling them apart.

**Writes** &nbsp; Nothing.

In [ ]:
import pandas as pd
pd.read_csv('results/nli_train.csv')

**Prints** &nbsp; Five browser downloads, or the files printed inline when that is unavailable.

**Writes** &nbsp; **Your machine**, finally — as downloads, or as text in the saved notebook. These five are what gets committed. The weights are not among them.

In [ ]:
# Browser download is Colab-frontend only. Under VS Code, print the contents instead so
# the numbers can be copied straight out of the saved notebook.
RESULTS = ['results/nli_train.csv', 'results/nli_train_alephbert.json',
           'results/nli_data_train.json', 'results/nli_data_val.json',
           'results/nli_data_test.json']
try:
    from google.colab import files
    for path in RESULTS:
        files.download(path)
except Exception as exc:
    print(f'[warn] browser download unavailable ({type(exc).__name__}) - contents below')
    for path in RESULTS:
        print(f'\n===== {path} =====')
        print(open(path, encoding='utf-8').read())

Commit the result files from your machine with the `nli:` prefix.

**Still open after this notebook:** `nli_rerank.py` now has a `pair` encoding mode and
reads its labels from `config.id2label`, so it *can* load this checkpoint — but `lam`
still defaults to 1.0, meaning pure NLI with the embedder's cosine contributing nothing.
Tuning λ on the probe's train split is the next piece.